# 01 · Download ISIMIP inputs

Downloads an approved, fixed URL manifest with SHA-256 checks. No automatic discovery or source selection is performed.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
#!/usr/bin/env python3
"""Download files from an approved ISIMIP URL manifest, with checksum audit.

Manifest columns: url, filename, sha256 (sha256 optional, but recommended).
The original ISIMIP queries must be documented when preparing the manifest.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
from pathlib import Path
from urllib.parse import urlparse
import requests


def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for part in iter(lambda:f.read(4*1024*1024),b''):h.update(part)
    return h.hexdigest()


def valid_manifest(path):
    with path.open(newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f)
        if not {'url','filename'}.issubset(reader.fieldnames or []):
            raise ValueError('Manifest must contain url and filename columns')
        items=list(reader)
    if not items:raise ValueError('Empty download manifest')
    seen=set()
    for row in items:
        parsed=urlparse(row['url'].strip())
        if parsed.scheme!='https' or not parsed.netloc:
            raise ValueError(f'Use an approved HTTPS source URL: {row["url"]}')
        name=row['filename'].strip()
        if not name or Path(name).name!=name or name in seen:
            raise ValueError(f'Invalid or duplicate destination filename: {name!r}')
        seen.add(name)
        sha=(row.get('sha256') or '').strip().lower()
        if sha and (len(sha)!=64 or any(c not in '0123456789abcdef' for c in sha)):
            raise ValueError(f'Invalid SHA256 for {name}')
    return items


def download(row, folder):
    dest=folder/row['filename'].strip()
    expected=(row.get('sha256') or '').strip().lower()
    if dest.is_file():
        got=digest(dest)
        if expected and got!=expected:raise ValueError(f'Existing {dest} checksum mismatch')
        print(f'EXISTS {dest.name} SHA256={got}');return
    part=dest.with_name(dest.name+'.part')
    if part.exists():raise FileExistsError(f'Partial file already exists: {part}')
    with requests.get(row['url'].strip(),stream=True,timeout=(20,120)) as r:
        r.raise_for_status()
        with part.open('wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:f.write(chunk)
    got=digest(part)
    if expected and got!=expected:
        part.unlink()
        raise ValueError(f'SHA256 mismatch for {dest.name}; source may have changed')
    part.rename(dest)
    print(f'DOWNLOADED {dest.name} SHA256={got}')


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
MANIFEST = ROOT / 'docs' / 'isimip-download-manifest.csv'
DOWNLOAD_ROOT = DATA / 'isimip'
RUN_DOWNLOAD = False  # Change to True only after reviewing the manifest URLs and licences.


## Execute and inspect

In [ ]:
rows = valid_manifest(MANIFEST)
print(f'Approved URL manifest: {len(rows)} files')
for row in rows:
    print(row['filename'], row['url'])
if RUN_DOWNLOAD:
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    for row in rows:
        download(row, DOWNLOAD_ROOT)
    print('STATUS: PASS_DOWNLOADS_HASHED')
else:
    print('STATUS: PASS_DOWNLOAD_MANIFEST_PREFLIGHT_ONLY')
